# Module 03: Model Development

**What you'll learn:**
- Why you should always start with baselines
- How to build and compare Linear, XGBoost, and LSTM models
- How to interpret model predictions and feature importance
- The BaseForecaster pattern for production ML

**Time:** ~2 hours

## 1. The Model Development Strategy

In production ML, you **never** jump straight to a complex model. The progression is:

```
Baseline → Simple ML → Advanced ML → Deep Learning
(naive)    (linear)    (XGBoost)     (LSTM)
```

**Why?** Baselines tell you if complex models are worth the effort.
- If a simple average beats your neural network → something is wrong with your data/features
- If XGBoost is only 2% better than linear → maybe the simpler model is better for production (faster, easier to debug)
- Each step up in complexity should justify itself with measurably better results

## 2. Setup

Let's prepare our data (same as previous notebooks):

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from energy_forecast.data.synthetic import SyntheticDataGenerator
from energy_forecast.features.engineering import FeatureEngineer
from energy_forecast.evaluation.metrics import MetricsCalculator
from sklearn.preprocessing import StandardScaler

# Generate data
gen = SyntheticDataGenerator(num_buildings=3, start_date='2023-01-01', end_date='2023-12-31', random_seed=42)
df = gen.generate()

# Engineer features
engineer = FeatureEngineer()
df = engineer.create_time_features(df)
df = engineer.create_lag_features(df, 'energy_demand_kwh', [1, 2, 3, 6, 12, 24, 168])
df = engineer.create_rolling_features(df, 'energy_demand_kwh', [6, 12, 24])
df = engineer.create_weather_features(df)
df = engineer.create_calendar_features(df)
df = df.dropna()

feature_names = engineer.get_feature_names(df)
X = df[feature_names].values
y = df['energy_demand_kwh'].values

# Time-based split
n = len(X)
X_train, y_train = X[:int(n*0.7)], y[:int(n*0.7)]
X_val, y_val = X[int(n*0.7):int(n*0.85)], y[int(n*0.7):int(n*0.85)]
X_test, y_test = X[int(n*0.85):], y[int(n*0.85):]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(f'Features: {len(feature_names)}')
print(f'Train: {X_train_s.shape}, Val: {X_val_s.shape}, Test: {X_test_s.shape}')

## 3. Baselines — Your Sanity Check

A **baseline** is the simplest possible prediction. Any real model must beat it to justify its existence.

Let's build three naive baselines:

In [ ]:
# Baseline 1: Predict the mean
mean_pred = np.full_like(y_test, y_train.mean())
mean_metrics = MetricsCalculator.compute_all(y_test, mean_pred)

# Baseline 2: Predict last known value (24 hours ago)
# Use the lag_24 feature from test data as a simple predictor
lag24_idx = feature_names.index('energy_demand_kwh_lag_24') if 'energy_demand_kwh_lag_24' in feature_names else 0
lag24_pred = X_test[:, lag24_idx]  # raw (unscaled) lag values
lag24_metrics = MetricsCalculator.compute_all(y_test, lag24_pred)

# Baseline 3: Rolling 7-day average
rolling_idx = [i for i, f in enumerate(feature_names) if 'rolling_mean_24' in f]
if rolling_idx:
    rolling_pred = X_test[:, rolling_idx[0]]
    rolling_metrics = MetricsCalculator.compute_all(y_test, rolling_pred)
else:
    rolling_metrics = {'rmse': 999, 'mae': 999, 'r2': 0}

print(f"{'Baseline':<25} {'RMSE':<10} {'MAE':<10} {'R2':<10}")
print('-' * 55)
print(f"{'Mean prediction':<25} {mean_metrics['rmse']:<10.2f} {mean_metrics['mae']:<10.2f} {mean_metrics['r2']:<10.4f}")
print(f"{'Last 24h value':<25} {lag24_metrics['rmse']:<10.2f} {lag24_metrics['mae']:<10.2f} {lag24_metrics['r2']:<10.4f}")
print(f"{'Rolling 24h average':<25} {rolling_metrics['rmse']:<10.2f} {rolling_metrics['mae']:<10.2f} {rolling_metrics['r2']:<10.4f}")
print('\nAny real model must beat these numbers!')

## 4. Linear Models — The First Real Model

### What is Ridge Regression?

It's linear regression with a twist: it adds a **penalty** for large coefficients.

- Regular linear regression: find weights that minimize prediction error
- Ridge regression: find weights that minimize error AND keep weights small

The `alpha` parameter controls how much to penalize: higher alpha = simpler model.

Why use it? It's **fast, interpretable, and hard to overfit**. The perfect first model.

In [ ]:
from energy_forecast.models.linear import LinearForecaster

model_linear = LinearForecaster(model_type='ridge', alpha=1.0)
metrics_linear = model_linear.fit(X_train_s, y_train, X_val=X_val_s, y_val=y_val)
pred_linear = model_linear.predict(X_test_s)
metrics_linear = MetricsCalculator.compute_all(y_test, pred_linear)

print('Ridge Regression Results:')
for k, v in metrics_linear.items():
    print(f'  {k}: {v:.4f}')

# Plot predictions
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test[:300], label='Actual', alpha=0.8)
ax.plot(pred_linear[:300], label='Predicted', alpha=0.8)
ax.legend()
ax.set_title('Linear Model: Actual vs Predicted (first 300 hours)')
ax.set_ylabel('Energy Demand (kWh)')
plt.tight_layout()
plt.show()

## 5. XGBoost — Gradient Boosting

### What is Gradient Boosting?

Imagine training **500 small decision trees**, where each tree tries to **fix the mistakes** of all the previous ones.

- Tree 1 makes predictions → has errors
- Tree 2 focuses on those errors → reduces them
- Tree 3 focuses on remaining errors → reduces them more
- ... and so on for 500 trees

It's like a team of students where each one specializes in the questions the others got wrong.

**Key hyperparameters:**
- `n_estimators`: How many trees (more = more capacity, slower)
- `max_depth`: How complex each tree can be (deeper = more flexible)
- `learning_rate`: How much each tree contributes (smaller = more conservative, needs more trees)

In [ ]:
from energy_forecast.models.xgboost_model import XGBoostForecaster

model_xgb = XGBoostForecaster(n_estimators=500, max_depth=6, learning_rate=0.05)
metrics_xgb_train = model_xgb.fit(X_train_s, y_train, X_val=X_val_s, y_val=y_val)
pred_xgb = model_xgb.predict(X_test_s)
metrics_xgb = MetricsCalculator.compute_all(y_test, pred_xgb)

print('XGBoost Results:')
for k, v in metrics_xgb.items():
    print(f'  {k}: {v:.4f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(y_test[:300], label='Actual', alpha=0.8)
axes[0].plot(pred_xgb[:300], label='Predicted', alpha=0.8)
axes[0].legend()
axes[0].set_title('XGBoost: Actual vs Predicted')

# Feature importance
importance = model_xgb.get_feature_importance()
top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]
axes[1].barh([f[0][:20] for f in top_features], [f[1] for f in top_features], color='steelblue')
axes[1].set_title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()

## 6. LSTM — Deep Learning for Time Series

### What is an LSTM?

Regular models see each data point **independently**. An LSTM sees **sequences** — it has memory.

**Analogy**: Reading a book word by word vs. reading it with memory of the plot.

For energy forecasting: to predict 3 PM demand, it helps to know what happened at 1 PM and 2 PM — the LSTM remembers this.

**How it works:**
1. We create "windows" of history: `[hour1, hour2, ..., hour24]` → predict `hour25`
2. The LSTM processes the window sequentially, building up context
3. It outputs a single prediction based on the full history

*Note: LSTM is slower to train. We'll use small settings here.*

In [ ]:
from energy_forecast.models.lstm_model import LSTMForecaster

model_lstm = LSTMForecaster(
    input_size=X_train_s.shape[1],
    hidden_size=64,
    num_layers=1,
    epochs=10,        # Small for demo
    sequence_length=24,
    batch_size=64,
    learning_rate=0.001
)
metrics_lstm_train = model_lstm.fit(X_train_s, y_train, X_val=X_val_s, y_val=y_val)

print('LSTM Training:')
for k, v in metrics_lstm_train.items():
    print(f'  {k}: {v:.4f}')

# Plot training loss
if model_lstm.training_losses:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(model_lstm.training_losses, label='Train Loss')
    if model_lstm.val_losses:
        ax.plot(model_lstm.val_losses, label='Val Loss')
    ax.legend()
    ax.set_title('LSTM Training Progress')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    plt.show()

## 7. Model Comparison

Let's compare all models fairly on the same test set:

In [ ]:
results = {
    'Mean Baseline': mean_metrics,
    'Linear (Ridge)': metrics_linear,
    'XGBoost': metrics_xgb,
}

print(f"{'Model':<20} {'RMSE':<10} {'MAE':<10} {'MAPE':<10} {'R2':<10}")
print('=' * 60)
for name, m in results.items():
    print(f"{name:<20} {m['rmse']:<10.4f} {m['mae']:<10.4f} {m.get('mape',0):<10.4f} {m['r2']:<10.4f}")

# Visual comparison
fig, ax = plt.subplots(figsize=(10, 5))
models = list(results.keys())
rmses = [results[m]['rmse'] for m in models]
colors = ['gray', 'steelblue', 'coral']
ax.bar(models, rmses, color=colors)
ax.set_ylabel('RMSE (lower is better)')
ax.set_title('Model Comparison')
for i, v in enumerate(rmses):
    ax.text(i, v + 0.5, f'{v:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. The BaseForecaster Pattern

Notice something? All our models share the same interface:

```python
model.fit(X_train, y_train, X_val, y_val)  # Train
model.predict(X_test)                       # Predict
model.save(path)                            # Save
model.load(path)                            # Load
model.get_params()                          # Get configuration
```

This is the **BaseForecaster** abstract class pattern. It means:
- Our training pipeline works with ANY model type
- Our serving API doesn't care if it's Linear, XGBoost, or LSTM
- Swapping models is a one-line config change

This is a critical production pattern: **decouple your infrastructure from your model choice**.

## 9. Exercises

1. **Hyperparameter sweep**: Try XGBoost with `max_depth` values of 3, 6, and 9. Which is best?
2. **Visual analysis**: Plot predictions for a single week (168 hours). Which model captures daily patterns best?
3. **Error analysis**: Calculate RMSE by hour-of-day for each model. When does each model struggle?

## 10. Key Takeaways

- **Always start with baselines** — they set the bar for what "good" means
- **Linear → XGBoost → LSTM** is a natural progression
- **More complex ≠ always better** — simpler models are easier to debug and deploy
- A **common model interface** enables modular, swappable pipelines
- Feature importance tells you what the model actually learned

**Next: [Notebook 04 - Training Pipelines](./04_training_pipelines.ipynb)**